In [15]:

import pandas as pd
from tqdm import tqdm
from transformers import pipeline
import joblib

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

df = pd.read_csv('/Users/sidhaanthkapoor/MentalHealthClassifier/data/raw/Combined_Data.csv')
df.head()

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [17]:
#basic preprocessing

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_text"] = df["statement"].apply(clean_text)

In [18]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['status'])

print(encoder.classes_)

['Anxiety' 'Bipolar' 'Depression' 'Normal' 'Personality disorder' 'Stress'
 'Suicidal']


In [19]:
#train test split (0.7/0.3)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['label'], test_size=0.3, random_state=42, stratify=df['label'])

In [20]:
#tfidfvectorizer, look for optimized parameters (this step is not complete yet, used gpt to suggest parameters for tfidf)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3, max_df=0.9, sublinear_tf=True)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [21]:
#logreg training

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(C=1, class_weight="balanced", max_iter=1000, multi_class='multinomial')

model.fit(X_train_tfidf, y_train)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(C=1, class_weight='balanced', max_iter=1000,
                   multi_class='multinomial')

In [22]:
#predictions

y_pred = model.predict(X_test_tfidf)

y_prob = model.predict_proba(X_test_tfidf)

In [24]:
#eval

from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred,target_names=encoder.classes_))

from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

                      precision    recall  f1-score   support

             Anxiety       0.78      0.80      0.79      1167
             Bipolar       0.79      0.78      0.78       863
          Depression       0.80      0.64      0.71      4621
              Normal       0.89      0.92      0.91      4905
Personality disorder       0.56      0.78      0.65       360
              Stress       0.51      0.71      0.59       801
            Suicidal       0.68      0.75      0.71      3196

            accuracy                           0.77     15913
           macro avg       0.72      0.77      0.73     15913
        weighted avg       0.78      0.77      0.77     15913

[[ 936   20   32   62   37   76    4]
 [  24  670   50   21   45   40   13]
 [ 115   98 2939  213   65  133 1058]
 [  38   19   33 4510   23  241   41]
 [   9    6   18   13  280   29    5]
 [  66   24   48   36   43  568   16]
 [   8   13  551  204    5   33 2382]]


In [27]:
import numpy as np

print(encoder.classes_)
print(np.unique(y_train))
print(np.unique(y_pred))

labels, counts = np.unique(y_pred, return_counts=True)

print(labels)
print(counts)

print(X_train_tfidf.shape)

print(pd.Series(y_pred).value_counts())

joblib.dump(model, '/Users/sidhaanthkapoor/MentalHealthClassifier/models/logreg_model.pkl')
joblib.dump(vectorizer, "/Users/sidhaanthkapoor/MentalHealthClassifier/models/tfidf_vectorizer.pkl")

['Anxiety' 'Bipolar' 'Depression' 'Normal' 'Personality disorder' 'Stress'
 'Suicidal']
[0 1 2 3 4 5 6]
[0 1 2 3 4 5 6]
[0 1 2 3 4 5 6]
[1196  850 3671 5059  498 1120 3519]
(37130, 20000)
3    5059
2    3671
6    3519
0    1196
5    1120
1     850
4     498
Name: count, dtype: int64


['/Users/sidhaanthkapoor/MentalHealthClassifier/models/tfidf_vectorizer.pkl']

In [13]:
#indicators

feature_names = vectorizer.get_feature_names_out()

for i, c in enumerate(encoder.classes_):
    top = model.coef_[i].argsort()[-20:]

    print(c)
    print(feature_names[top])

Anxiety
['disease' 'panic' 'my anxiety' 'freaking' 'fear' 'blood' 'worrying'
 'scared' 'heart' 'health' 'symptoms' 'health anxiety' 'restlessness'
 'cancer' 'worry' 'nervous' 'worried' 'anxious' 'restless' 'anxiety']
Bipolar
['my meds' 'medication' 'episodes' 'nan' 'bp' 'mood' 'illness' 've'
 'hypomania' 'seroquel' 'stable' 'latuda' 'hypomanic' 'lithium' 'lamictal'
 'mania' 'episode' 'meds' 'manic' 'bipolar']
Depression
['feel' 'parent' 'na' 'do' 'just' 'friend' 'did not' 'life' 'not know'
 'it is' 'pression' 'doe' 'ha' 'not' 'cannot' 'depressed' 'am' 'do not'
 'wa' 'depression']
Normal
['account' 'holiday' 'met' 'tweet' 'mutual' 'or not' 'have to' 'morning'
 'haha' 'ha' 'we' 'he' 'her' 'twitter' 'eid' 'quot' 'ðÿ' 'yes' 'url' 'wa']
Personality disorder
['shame' 'is your' 'hypochondria' 'my avpd' 'personality' 'this disorder'
 'avoiding' 'view' 'interaction' 'with avpd' 'do you' 'avoid' 'view poll'
 'poll' 'don' 'social' 'nan' 'people' 'avoidant' 'avpd']
Stress
['stressful' 'relax' 'pay

In [7]:
print(df["status"].value_counts())

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64
